# d45 data structure update

This notebook demonstrates the updated FRUST dataframe contract without running xTB or ORCA. It focuses on parsing, column names, old-data normalization, and the new `lowest` grouping semantics.

In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

from frust.stepper import Stepper
from frust.schema import energy_columns, infer_group_columns, normalize_dataframe

## A tiny embedded molecule

The examples below use one methane conformer only so we can inspect dataframe behavior without doing real chemistry.

In [2]:
mol = Chem.AddHs(Chem.MolFromSmiles("C"))
AllChem.EmbedMolecule(mol, randomSeed=1)
cids = [0]

step = Stepper(save_output_dir=False)

2026-04-30 10:28:23 INFO  frust.stepper.GENERIC.job1111: Working dir: .


## TS name parsing preserves substrate names

The old parser used underscore splitting and could turn `methyl_furan-2-carboxylate` into `furan-2-carboxylate`. The fallback parser now treats the wrapped TS name as a legacy display name and extracts structured metadata conservatively.

In [3]:
ts_embedded = {
    "TS1(methyl_furan-2-carboxylate_rpos(4))": (
        mol,
        cids,
        [0, 1, 2, 3, 4, 5],
        "COC(=O)c1ccco1",
        [],
    )
}

ts_df = step.build_initial_df(ts_embedded)
ts_df[[
    "structure_id",
    "custom_name",
    "substrate_name",
    "structure_type",
    "molecule_role",
    "rpos",
    "constraint_atoms",
]]

,structure_id,custom_name,substrate_name,structure_type,molecule_role,rpos,constraint_atoms
0,TS1:methyl_furan-2-carboxylate:r4,TS1(methyl_furan-2-carboxylate_rpos(4)),methyl_furan-2-carboxylate,TS1,ts,4,"[0, 1, 2, 3, 4, 5]"


In [4]:
ts_df

,structure_id,custom_name,substrate_name,structure_type,molecule_role,rpos,constraint_atoms,cid,smiles,input_smiles,atoms,coords_embedded,energy_uff
0,TS1:methyl_furan-2-carboxylate:r4,TS1(methyl_furan-2-carboxylate_rpos(4)),methyl_furan-2-carboxylate,TS1,ts,4,"[0, 1, 2, 3, 4, 5]",0,COC(=O)c1ccco1,COC(=O)c1ccco1,"[C, H, H, H, H]","[(0.005118712612069832, -0.010620498663889944,...",None


## Explicit metadata for non-TS structures

New generated molecule records can carry metadata directly. The dataframe no longer needs to guess that `int2` is a molecule role and not a substrate.

In [5]:
def record(key, role, rpos=None):
    suffix = f":r{rpos}" if rpos is not None else ""
    return (
        mol,
        cids,
        {
            "structure_id": f"MOL:1-methylpyrrole:{role}{suffix}",
            "custom_name": key,
            "substrate_name": "1-methylpyrrole",
            "input_smiles": "CN1C=CC=C1",
            "smiles": "CN1C=CC=C1",
            "structure_type": "MOL",
            "molecule_role": role,
            "rpos": rpos,
        },
    )

mols_embedded = {
    "CN1C=CC=C1_1-methylpyrrole": record("CN1C=CC=C1_1-methylpyrrole", "ligand"),
    "CN1C=CC=C1_int2_rpos(2)": record("CN1C=CC=C1_int2_rpos(2)", "int2", 2),
    "CN1C=CC=C1_mol2_rpos(2)": record("CN1C=CC=C1_mol2_rpos(2)", "mol2", 2),
}

mols_df = step.build_initial_df(mols_embedded)
mols_df[["custom_name", "substrate_name", "structure_type", "molecule_role", "rpos", "constraint_atoms"]]

,custom_name,substrate_name,structure_type,molecule_role,rpos,constraint_atoms
0,CN1C=CC=C1_1-methylpyrrole,1-methylpyrrole,MOL,ligand,<NA>,None
1,CN1C=CC=C1_int2_rpos(2),1-methylpyrrole,MOL,int2,2,None
2,CN1C=CC=C1_mol2_rpos(2),1-methylpyrrole,MOL,mol2,2,None


## Short calculation suffixes

Old result suffixes normalize to the compact schema: `electronic_energy -> EE`, `normal_termination -> NT`, `opt_coords -> oc`, and `gibbs_energy -> GE`.

In [6]:
old = pd.DataFrame(
    {
        "ligand_name": ["legacy name"],
        "DFT-wB97X-D3-6-31G**-OptTS-electronic_energy": [-123.4],
        "DFT-wB97X-D3-6-31G**-OptTS-normal_termination": [True],
        "DFT-wB97X-D3-6-31G**-OptTS-opt_coords": [[[0.0, 0.0, 0.0]]],
        "DFT-wB97X-D3-6-31G**-OptTS-gibbs_energy": [-122.9],
    }
)

normalized = normalize_dataframe(old)
normalized

,substrate_name,DFT-wB97X-D3-6-31G**-OptTS-EE,DFT-wB97X-D3-6-31G**-OptTS-NT,DFT-wB97X-D3-6-31G**-OptTS-oc,DFT-wB97X-D3-6-31G**-OptTS-GE
0,legacy name,-123.4,True,"[[0.0, 0.0, 0.0]]",-122.9


New calculation columns are intended to be stable and readable when callers give explicit step names, for example `xtb_preopt-EE`, `xtb_preopt-NT`, `xtb_preopt-oc`, and `dft_tsopt-GE`.

In [7]:
fake_results = mols_df.copy()
fake_results["xtb_preopt-EE"] = [-10.0, -40.0, -30.0]
fake_results["xtb_preopt-NT"] = [True, True, True]
fake_results["xtb_preopt-oc"] = fake_results["coords_embedded"]

fake_results[["substrate_name", "molecule_role", "rpos", "xtb_preopt-EE", "xtb_preopt-NT", "xtb_preopt-oc"]]

,substrate_name,molecule_role,rpos,xtb_preopt-EE,xtb_preopt-NT,xtb_preopt-oc
0,1-methylpyrrole,ligand,<NA>,-10.0,True,"[(0.005118712612069832, -0.010620498663889944,..."
1,1-methylpyrrole,int2,2,-40.0,True,"[(0.005118712612069832, -0.010620498663889944,..."
2,1-methylpyrrole,mol2,2,-30.0,True,"[(0.005118712612069832, -0.010620498663889944,..."


## Why `lowest` grouping changed

Grouping only by substrate and rpos makes `int2` and `mol2` compete against each other at the same rpos. Grouping by substrate, structure type, molecule role, and rpos keeps the lowest conformer for each actual chemical object.

In [8]:
comparison = pd.DataFrame(
    {
        "substrate_name": ["pyrrole", "pyrrole", "pyrrole", "pyrrole"],
        "structure_type": ["MOL", "MOL", "MOL", "MOL"],
        "molecule_role": ["int2", "int2", "mol2", "mol2"],
        "rpos": [2, 2, 2, 2],
        "cid": [0, 1, 0, 1],
        "xtb_preopt-EE": [-40.0, -42.0, -30.0, -33.0],
    }
)

energy_col = energy_columns(comparison)[-1]
new_group_cols = infer_group_columns(comparison)

old_lowest = (
    comparison.sort_values(["substrate_name", "rpos", energy_col])
    .groupby(["substrate_name", "rpos"], dropna=False)
    .head(1)
)

new_lowest = (
    comparison.sort_values(new_group_cols + [energy_col])
    .groupby(new_group_cols, dropna=False)
    .head(1)
)

old_lowest, new_lowest

(  substrate_name structure_type molecule_role  rpos  cid  xtb_preopt-EE
 1        pyrrole            MOL          int2     2    1          -42.0,
   substrate_name structure_type molecule_role  rpos  cid  xtb_preopt-EE
 1        pyrrole            MOL          int2     2    1          -42.0
 3        pyrrole            MOL          mol2     2    1          -33.0)

In the tuple above, `old_lowest` keeps only the best row across both `int2` and `mol2`. `new_lowest` keeps one `int2` row and one `mol2` row, which is the behavior FRUST wants for mixed dataframe workflows.